In [63]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from src.train_pipeline import build_pipeline,train_and_evaluate
from src.config import clean_data_output_path, TARGET_COL
import numpy as np


In [64]:
preprocessor, X, y = build_pipeline()
X_train, X_test, y_train, y_test = train_and_evaluate(preprocessor, X, y)

Accuracy: 0.744
Classification Report:               precision    recall  f1-score   support

           0       0.67      0.97      0.79      1120
           1       0.95      0.52      0.67      1130

    accuracy                           0.74      2250
   macro avg       0.81      0.75      0.73      2250
weighted avg       0.81      0.74      0.73      2250

Returning train/test splits


In [65]:
X_train_t = torch.tensor(np.array(X_train), dtype=torch.float32)
y_train_t = torch.tensor(np.array(y_train), dtype=torch.long)
X_test_t = torch.tensor(np.array(X_test), dtype=torch.float32)
y_test_t = torch.tensor(np.array(y_test), dtype=torch.long)
input_dim = X_train_t.shape[1]
output_dim = len(torch.unique(y_train_t))

In [66]:
class HeartANN(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, output_dim=2):
        super(HeartANN, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim//2)
        self.fc3 = nn.Linear(hidden_dim//2, hidden_dim//2)
        self.fc4 = nn.Linear(hidden_dim//2, output_dim)
    def forward(self, x):
        x= self.relu(self.fc1(x))
        x= self.relu(self.fc2(x))
        x= self.relu(self.fc3(x))
        x= self.fc4(x)
        return x

In [85]:
model= HeartANN(input_dim, hidden_dim=64, output_dim=output_dim)
loss_function= nn.CrossEntropyLoss()
optimizer= optim.Adam(model.parameters(), lr=0.001)
num_epochs= 500
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    outputs= model(X_train_t)
    loss= loss_function(outputs, y_train_t)
    loss.backward()
    optimizer.step()
    pred=torch.argmax(outputs, dim=1)
    acc= accuracy_score(y_train_t, pred)
    if (epoch+1) % 10 ==0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Accuracy: {acc:.4f}')
    


Epoch [10/500], Loss: 0.6921, Accuracy: 0.5030
Epoch [20/500], Loss: 0.6897, Accuracy: 0.5442
Epoch [30/500], Loss: 0.6867, Accuracy: 0.5503
Epoch [40/500], Loss: 0.6826, Accuracy: 0.5571
Epoch [50/500], Loss: 0.6771, Accuracy: 0.5716
Epoch [60/500], Loss: 0.6692, Accuracy: 0.5936
Epoch [70/500], Loss: 0.6587, Accuracy: 0.6144
Epoch [80/500], Loss: 0.6456, Accuracy: 0.6338
Epoch [90/500], Loss: 0.6300, Accuracy: 0.6518
Epoch [100/500], Loss: 0.6134, Accuracy: 0.6703
Epoch [110/500], Loss: 0.5970, Accuracy: 0.6822
Epoch [120/500], Loss: 0.5808, Accuracy: 0.6949
Epoch [130/500], Loss: 0.5661, Accuracy: 0.7070
Epoch [140/500], Loss: 0.5522, Accuracy: 0.7209
Epoch [150/500], Loss: 0.5387, Accuracy: 0.7336
Epoch [160/500], Loss: 0.5267, Accuracy: 0.7417
Epoch [170/500], Loss: 0.5156, Accuracy: 0.7493
Epoch [180/500], Loss: 0.5046, Accuracy: 0.7586
Epoch [190/500], Loss: 0.4949, Accuracy: 0.7644
Epoch [200/500], Loss: 0.4866, Accuracy: 0.7711
Epoch [210/500], Loss: 0.4784, Accuracy: 0.7728
E

In [86]:
model.eval()
with torch.no_grad():
    outputs = model(X_test_t)
    preds = torch.argmax(outputs, dim=1)

acc = accuracy_score(y_test_t, preds)
print("Test Accuracy:", acc)
print("Classification Report:\n", classification_report(y_test_t, preds))


Test Accuracy: 0.6262222222222222
Classification Report:
               precision    recall  f1-score   support

           0       0.63      0.61      0.62      1120
           1       0.63      0.64      0.63      1130

    accuracy                           0.63      2250
   macro avg       0.63      0.63      0.63      2250
weighted avg       0.63      0.63      0.63      2250

